# Setup

In [ ]:
# Optional autoreload while developing locally

%load_ext autoreload
%autoreload 2

In [ ]:

# 0) Local-only setup and base diagnostics
import os
import sys
import platform
from pathlib import Path

ENV = "Local"

print("ENV:", ENV)
print("Python:", sys.version)
print("Platform:", platform.platform())
print("CUDA visible devices:", os.environ.get("CUDA_VISIBLE_DEVICES"))

required_files = {
    "tokenizer.py",
    "doc_chunker.py",
    "coreference_cluster_extractor.py",
    "local_clusters.py",
    "artifact_io.py",
}

PROJECT_DIR = None
local_candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]

for candidate in local_candidates:
    if candidate.exists() and required_files.issubset(
        {x.name for x in candidate.iterdir() if x.is_file()}
    ):
        PROJECT_DIR = candidate
        break

if PROJECT_DIR is None:
    PROJECT_DIR = Path.cwd()
    print(
        "[warning] Could not find all project .py files in cwd/parents. "
        "Using current working directory anyway:", PROJECT_DIR
    )

sys.path.insert(0, str(PROJECT_DIR))
print("PROJECT_DIR =", PROJECT_DIR)
print("sys.path[0] =", sys.path[0])
print("Project files found:")
for name in sorted(required_files):
    path = PROJECT_DIR / name
    print(" -", name, "OK" if path.exists() else "MISSING")


In [ ]:

# 0.3) Shared device diagnostics
import torch

print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU count:", torch.cuda.device_count())
    print("GPU 0:", torch.cuda.get_device_name(0))
    DEVICE = "cuda:0"
else:
    DEVICE = "cpu"
print("Using DEVICE =", DEVICE)


# Imports

In [ ]:

import pickle
import re

import pandas as pd

from tokenizer import create_tokenizer, ensure_booknlp_extensions, tokens_to_dicts
from doc_chunker import create_chunker
from coreference_cluster_extractor import create_coreference_cluster_extractor
from local_clusters import (
    chunk_specs_to_rows,
    create_local_clusters_metadata,
    extracted_chunk_to_rows,
    validate_chunk_rows,
)
from artifact_io import LocalClusterMarathonStore, load_local_clusters_artifact

from tokenizer import create_tokenizer, ensure_booknlp_extensions, tokens_to_dicts

from coref_schema import register_spacy_coref_extension, require_coref_layer
from annotator import annotate_doc_with_global_coref

import torch
import torch.serialization as torch_serialization


In [ ]:
# from global_coref_merger import merge_local_coreference_clusters # -> connected-components version
from global_coref_merger_aggregative import merge_local_coreference_clusters # -> aggregative version

from mention_filter import (build_anchor_bank, filter_mentions_by_foreign_anchors,load_generic_anchor_keys)

# Functions

In [ ]:
# text loading + preprocessing
def load_txt(path):
    path = Path(path)

    try:
        return path.read_text(encoding="utf-8")
    except UnicodeDecodeError:
        return path.read_text(encoding="latin-1")

In [ ]:
def preprocess_for_NER(text):
    # Normalize Windows/Mac newlines
    text = text.replace("\r\n", "\n").replace("\r", "\n")

    # Remove possible invisible BOM
    text = text.replace("\ufeff", "")

    # Normalize curly apostrophes/quotes
    text = text.replace("’", "'")
    text = text.replace("‘", "'")
    text = text.replace("“", '"').replace("”", '"')

    # Replace line breaks inside paragraphs with spaces,
    # but preserve paragraph boundaries
    paragraphs = text.split("\n\n")
    paragraphs = [
        re.sub(r"\s*\n\s*", " ", paragraph).strip()
        for paragraph in paragraphs
        if paragraph.strip()
    ]

    text = "\n\n".join(paragraphs)

    chapter_re = re.compile(
        r"(?im)^\s*chapter\s+(?:[ivxlcdm]+|\d+)\b[^\n]*\n*"
    )

    text = chapter_re.sub("", text)

    # Normalize multiple spaces
    text = re.sub(r"[ \t]+", " ", text)

    return text.strip()

In [ ]:

def resolve_text_path(
    *,
    project_dir: Path,
    filename: str = "oz_full.txt",
    local_path: str | Path = "../../datasets/libri/oz_full.txt",
) -> Path:
    """Resolve the local input text path."""
    text_path = Path(local_path)
    if text_path.exists():
        return text_path

    fallback_path = project_dir / filename
    if fallback_path.exists():
        return fallback_path

    raise FileNotFoundError(
        "Could not resolve a local text file. "
        f"Tried LOCAL_TEXT_PATH={text_path} and PROJECT_DIR / {filename!r} = {fallback_path}."
    )


In [ ]:

def save_doc(doc, path: str | Path) -> Path:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("wb") as f:
        pickle.dump(doc, f)
    return path


def load_doc(path: str | Path):
    # Custom spaCy extensions must be registered before unpickling the Doc.
    ensure_booknlp_extensions()
    register_spacy_coref_extension()
    path = Path(path)
    with path.open("rb") as f:
        return pickle.load(f)


def detect_runtime_profile(*, force_profile: str | None = None) -> dict:
    """Return one explicit local runtime profile."""
    import torch

    cuda_available = torch.cuda.is_available()
    gpu_count = torch.cuda.device_count() if cuda_available else 0
    gpu_name = torch.cuda.get_device_name(0) if cuda_available else ""

    if force_profile:
        kind = force_profile
    elif cuda_available:
        kind = "local_cuda"
    else:
        kind = "local_cpu"

    defaults = {
        "local_cpu": {"chunk_size": 3000, "overlap_sentences": 30},
        "local_cuda": {"chunk_size": 6000, "overlap_sentences": 60},
    }
    profile_defaults = defaults.get(kind, defaults["local_cpu"])

    return {
        "kind": kind,
        "env": "Local",
        "cuda_available": cuda_available,
        "gpu_count": gpu_count,
        "gpu_name": gpu_name,
        "device": "cuda:0" if cuda_available else "cpu",
        "cpu_load_first": bool(cuda_available),
        "precision_policy": "auto",
        "p100_fallback_to_float32": False,
        "default_chunk_size": profile_defaults["chunk_size"],
        "default_overlap_sentences": profile_defaults["overlap_sentences"],
    }


def memory_snapshot() -> dict:
    snapshot = {}
    try:
        import resource
        snapshot["max_rss_mb"] = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024
    except Exception:
        pass
    try:
        import torch
        if torch.cuda.is_available():
            snapshot["cuda_allocated_mb"] = torch.cuda.memory_allocated() / (1024 ** 2)
            snapshot["cuda_reserved_mb"] = torch.cuda.memory_reserved() / (1024 ** 2)
    except Exception:
        pass
    return snapshot


def cleanup_after_chunk(runtime_profile: dict) -> None:
    import gc

    gc.collect()
    if str(runtime_profile.get("device", "")).startswith("cuda"):
        try:
            import torch
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
                if hasattr(torch.cuda, "ipc_collect"):
                    torch.cuda.ipc_collect()
        except Exception:
            pass


In [ ]:
def patch_torch_load_weights_only_false() -> None:
    """
    Idempotently patch torch.load so PyTorch/Lightning checkpoint loading uses
    weights_only=False, without creating recursive wrappers when the cell is
    executed multiple times.
    """

    # Recover the real PyTorch implementation if torch.load was already patched.
    # In normal PyTorch, torch.load is exported from torch.serialization.load.
    torch.load = torch_serialization.load

    if not hasattr(torch, "_maverick_original_torch_load"):
        torch._maverick_original_torch_load = torch_serialization.load

    def _torch_load_force_weights_only_false(*args, **kwargs):
        kwargs["weights_only"] = False
        return torch._maverick_original_torch_load(*args, **kwargs)

    _torch_load_force_weights_only_false._maverick_weights_patch = True
    torch.load = _torch_load_force_weights_only_false

# Config

## I/O config

In [ ]:

# Requested local file
TEXT_FILENAME = "oz_full.txt"
LOCAL_TEXT_PATH = Path(f"../../datasets/libri/{TEXT_FILENAME}")

TEXT_PATH = resolve_text_path(
    project_dir=PROJECT_DIR,
    filename=TEXT_FILENAME,
    local_path=LOCAL_TEXT_PATH,
)

OUTPUT_ROOT = PROJECT_DIR / "outputs"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

TOKENIZED_DOC_PATH = OUTPUT_ROOT / "tokenized_doc.pkl"
LOCAL_CLUSTERS_ARTIFACT_PATH = OUTPUT_ROOT / "maverick_local_clusters.zip"
LOCAL_MARATHON_CACHE_DIR = OUTPUT_ROOT / ".maverick_local_cache"
ANNOTATED_DOC_PATH = OUTPUT_ROOT / "annotated_doc.pkl"

print("TEXT_PATH =", TEXT_PATH)
print("OUTPUT_ROOT =", OUTPUT_ROOT)
print("TOKENIZED_DOC_PATH =", TOKENIZED_DOC_PATH)
print("LOCAL_CLUSTERS_ARTIFACT_PATH =", LOCAL_CLUSTERS_ARTIFACT_PATH)
print("LOCAL_MARATHON_CACHE_DIR =", LOCAL_MARATHON_CACHE_DIR)
print("ANNOTATED_DOC_PATH =", ANNOTATED_DOC_PATH)

In [ ]:
GLOBAL_COREF_OUTPUT_DIR = OUTPUT_ROOT / "global_coref"
GLOBAL_COREF_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("GLOBAL_COREF_OUTPUT_DIR =", GLOBAL_COREF_OUTPUT_DIR)

## Runtime and pipeline config

In [ ]:

# Pipeline configuration
SPACY_MODEL = "en_core_web_sm"
TOKENIZER_DISABLE = ("ner",)

HF_NAME_OR_PATH = "sapienzanlp/maverick-mes-litbank"
SINGLETONS = True
CLEAN_MENTIONS = True
MAX_MENTION_TOKENS = 20
DROP_CROSS_SENTENCE_MENTIONS = True

# Local runtime profile. Set FORCE_RUNTIME_PROFILE to None, "local_cpu", or "local_cuda".
FORCE_RUNTIME_PROFILE = None
RUNTIME_PROFILE = detect_runtime_profile(force_profile=FORCE_RUNTIME_PROFILE)
DEVICE = RUNTIME_PROFILE["device"]

# CHUNK_SIZE means expanded chunk size, including overlap.
# If CHUNK_SIZE = 2000, every materialized DocChunk must contain <= 2000 tokens total.
CHUNK_SIZE = RUNTIME_PROFILE["default_chunk_size"]
OVERLAP_SENTENCES = RUNTIME_PROFILE["default_overlap_sentences"]
MAX_EXPANDED_CHUNK_TOKENS = CHUNK_SIZE

# Local marathon-cache behavior.
LOCAL_CLUSTER_EXECUTION_MODE = "marathon_cache"
FAIL_ON_INCOMPATIBLE_MANIFEST = True
RECOMPUTE_CORRUPT_CHUNKS = True
KEEP_CORRUPT_SHARD_BACKUPS = True
FORCE_RECOMPUTE_CHUNKS = set()
FORCE_RECOMPUTE_ALL_LOCAL_CLUSTERS = False
ENABLE_SHARD_CHECKSUMS = True
FINAL_STREAMING_VALIDATE_GLOBAL_IDS = True
DELETE_CACHE_AFTER_FINALIZE = False
OVERWRITE_EXISTING_FINAL_ZIP = True

# Per-chunk validation catches broken rows before committing a shard.
PER_CHUNK_VALIDATION_MODE = "basic"  # "basic" or "debug"

print("RUNTIME_PROFILE =", RUNTIME_PROFILE)
print("DEVICE =", DEVICE)
print("CHUNK_SIZE(expanded) =", CHUNK_SIZE)
print("OVERLAP_SENTENCES =", OVERLAP_SENTENCES)
print("MAX_EXPANDED_CHUNK_TOKENS =", MAX_EXPANDED_CHUNK_TOKENS)
print("LOCAL_CLUSTER_EXECUTION_MODE =", LOCAL_CLUSTER_EXECUTION_MODE)
print("LOCAL_MARATHON_CACHE_DIR =", LOCAL_MARATHON_CACHE_DIR)
print("PER_CHUNK_VALIDATION_MODE =", PER_CHUNK_VALIDATION_MODE)


# Main

### Tokenization

In [ ]:
# Load or create the stable tokenized Doc artifact.
if TOKENIZED_DOC_PATH.exists():
    print(f"[doc] Loading tokenized Doc from {TOKENIZED_DOC_PATH}")
    doc = load_doc(TOKENIZED_DOC_PATH)
else:
    print("[doc] Tokenized Doc not found; creating it from raw text.")
    raw_text = load_txt(TEXT_PATH)
    text = preprocess_for_NER(raw_text)
    tokenizer = create_tokenizer(
        SPACY_MODEL,
        disable=TOKENIZER_DISABLE,
        verbose=True,
    )
    doc = tokenizer.tokenize(text)
    save_doc(doc, TOKENIZED_DOC_PATH)
    print(f"[doc] Saved tokenized Doc to {TOKENIZED_DOC_PATH}")

print("Doc tokens:", len(doc))
print("Doc sentences:", sum(1 for _ in doc.sents))

In [ ]:
# df_tokens = pd.DataFrame(tokens_to_dicts(doc))
# df_tokens.head()

### Chunking

In [ ]:
chunker = create_chunker(
    chunk_size=CHUNK_SIZE,
    overlap_sentences=OVERLAP_SENTENCES,
    max_expanded_chunk_tokens=MAX_EXPANDED_CHUNK_TOKENS,
)
chunk_plan = chunker.plan(doc)

In [ ]:
# print(f"Planned {len(chunk_plan)} chunks without materializing chunk docs.")
# for spec in chunk_plan.specs[:5]:
#     print(
#         spec.chunk_id,
#         "expanded=", (spec.global_start, spec.global_end),
#         "expanded_tokens=", spec.n_tokens,
#         "core=", (spec.core_start, spec.core_end),
#         "left_overlap=", (spec.left_overlap_start, spec.left_overlap_end),
#         "right_overlap=", (spec.right_overlap_start, spec.right_overlap_end),
#     )


### Coreference clusters extraction

##### Local (chunk-level) coreference cluster extraction

In [ ]:
patch_torch_load_weights_only_false() # Recover and safely patch torch.load for Maverick / PyTorch checkpoint loading
print("torch.load safely patched:", getattr(torch.load, "_maverick_weights_patch", False))

In [ ]:
# 2) Maverick local extraction: local marathon cache, per-chunk shards, streaming final aggregation.
coref_extractor = create_coreference_cluster_extractor(
    hf_name_or_path=HF_NAME_OR_PATH,
    device=DEVICE,
    singletons=SINGLETONS,
    verbose=True,
    clean_mentions=CLEAN_MENTIONS,
    max_mention_tokens=MAX_MENTION_TOKENS,
    drop_cross_sentence_mentions=DROP_CROSS_SENTENCE_MENTIONS,
    cpu_load_first=RUNTIME_PROFILE["cpu_load_first"],
    precision_policy=RUNTIME_PROFILE["precision_policy"],
    p100_fallback_to_float32=RUNTIME_PROFILE["p100_fallback_to_float32"],
)

maverick_config = {
    "hf_name_or_path": HF_NAME_OR_PATH,
    "device": DEVICE,
    "singletons": SINGLETONS,
    "clean_mentions": CLEAN_MENTIONS,
    "max_mention_tokens": MAX_MENTION_TOKENS,
    "drop_cross_sentence_mentions": DROP_CROSS_SENTENCE_MENTIONS,
    "chunk_size": CHUNK_SIZE,
    "chunk_size_semantics": "expanded_chunk_tokens_including_overlap",
    "overlap_sentences": OVERLAP_SENTENCES,
    "max_expanded_chunk_tokens": MAX_EXPANDED_CHUNK_TOKENS,
    "runtime_profile": RUNTIME_PROFILE,
}

metadata = create_local_clusters_metadata(
    doc=doc,
    chunk_count=len(chunk_plan),
    chunk_size=CHUNK_SIZE,
    overlap_sentences=OVERLAP_SENTENCES,
    max_expanded_chunk_tokens=MAX_EXPANDED_CHUNK_TOKENS,
    maverick_config=maverick_config,
    document_id=TEXT_PATH.stem,
)

store = LocalClusterMarathonStore(
    cache_dir=LOCAL_MARATHON_CACHE_DIR,
    output_zip_path=LOCAL_CLUSTERS_ARTIFACT_PATH,
    metadata=metadata,
    chunk_plan=chunk_plan,
    maverick_config=maverick_config,
    recompute_corrupt_chunks=RECOMPUTE_CORRUPT_CHUNKS,
    keep_corrupt_shard_backups=KEEP_CORRUPT_SHARD_BACKUPS,
    force_recompute_chunks=FORCE_RECOMPUTE_CHUNKS,
    force_recompute_all=FORCE_RECOMPUTE_ALL_LOCAL_CLUSTERS,
    enable_checksums=ENABLE_SHARD_CHECKSUMS,
    final_streaming_validate_global_ids=FINAL_STREAMING_VALIDATE_GLOBAL_IDS,
    overwrite_existing_final_zip=OVERWRITE_EXISTING_FINAL_ZIP,
    delete_cache_after_finalize=DELETE_CACHE_AFTER_FINALIZE,
)

store.validate_or_initialize_run()

current_chunk_id = None
try:
    for spec in chunk_plan.specs:
        current_chunk_id = spec.chunk_id

        if not store.should_process_chunk(spec):
            print(f"[chunk][skip] {spec.chunk_id}")
            continue

        print(
            f"[chunk] Processing {spec.chunk_id} "
            f"({spec.chunk_index + 1}/{len(chunk_plan)}) "
            f"expanded_tokens={spec.n_tokens}"
        )
        store.mark_chunk_started(spec)

        chunk = chunker.materialize(doc, spec)
        extracted = coref_extractor.extract(chunk)
        chunk_rows = extracted_chunk_to_rows(chunk=chunk, extracted=extracted)

        chunk_validation_report = validate_chunk_rows(
            doc=doc,
            chunk=chunk,
            rows=chunk_rows,
            mode=PER_CHUNK_VALIDATION_MODE,
        )

        store.commit_chunk(
            spec=spec,
            rows=chunk_rows,
            validation_report={
                **chunk_validation_report,
                "memory_after_validation": memory_snapshot(),
            },
        )

        print(
            f"[chunk] {spec.chunk_id}: "
            f"clusters={len(chunk_rows.clusters_rows)}, "
            f"mentions={len(chunk_rows.mentions_rows)}"
        )

        del chunk_rows
        del extracted
        del chunk
        coref_extractor.clear_runtime_state()
        cleanup_after_chunk(RUNTIME_PROFILE)

    store.validate_all_shards_complete()
    artifact_path = store.finalize_streaming()
    print(f"Saved final artifact to {artifact_path}")

except Exception as exc:
    if current_chunk_id is not None:
        # Find the current spec without relying on partially materialized runtime objects.
        failing_spec = next((s for s in chunk_plan.specs if s.chunk_id == current_chunk_id), None)
        if failing_spec is not None:
            store.record_chunk_failure(spec=failing_spec, exc=exc)
    print("[failure] The final artifact was not created successfully.")
    print("[failure] Marathon cache directory:", LOCAL_MARATHON_CACHE_DIR)
    raise
finally:
    try:
        coref_extractor.clear_runtime_state()
    except Exception:
        pass
    cleanup_after_chunk(RUNTIME_PROFILE)


##### local (chunk-level) coreference cluster artifact finalization

In [ ]:

# 3) Optional artifact inspection without re-running Maverick
if not LOCAL_CLUSTERS_ARTIFACT_PATH.exists():
    raise FileNotFoundError(f"Final artifact does not exist yet: {LOCAL_CLUSTERS_ARTIFACT_PATH}")

loaded_tables = load_local_clusters_artifact(LOCAL_CLUSTERS_ARTIFACT_PATH)
print("Artifact metadata:")
print(loaded_tables.metadata)


In [ ]:
# Optional mention preview
pd.DataFrame(loaded_tables.clusters_rows)

##### Destructive filter

In [ ]:
from pathlib import Path
import sys
import pandas as pd

# If filter_mentions_foreign_anchor.py is in the same directory as the notebook,
# this is usually enough.
PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))



# ---- Config ----------------------------------------------------------------

DESTRUCTIVE_FILTER_ENABLED = True

DESTRUCTIVE_FILTER_OUTPUT_DIR = GLOBAL_COREF_OUTPUT_DIR / "destructive_mention_filter"
DESTRUCTIVE_FILTER_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FILTERED_MENTIONS_PATH = DESTRUCTIVE_FILTER_OUTPUT_DIR / "mentions.filtered.csv"
DROPPED_MENTIONS_PATH = DESTRUCTIVE_FILTER_OUTPUT_DIR / "mentions.dropped.csv"
SELECTED_ANCHORS_PATH = DESTRUCTIVE_FILTER_OUTPUT_DIR / "selected_anchors.csv"

ANCHOR_PERCENTILE = 0.95          # top 5%
MIN_ANCHOR_MENTIONS = 2
FUZZY_THRESHOLD = 0.86
OWN_FAMILY_FUZZY_THRESHOLD = 0.90
MIN_FUZZY_CHARS = 5

# Optional custom blacklist file.
# Use None to rely only on the built-in conservative generic blacklist.
GENERIC_ANCHOR_BLACKLIST_PATH = None


# ---- Small adapters ---------------------------------------------------------

def rows_to_dataframe(rows) -> pd.DataFrame:
    """
    Accept either a DataFrame or a list[dict]-style row collection.
    """
    if isinstance(rows, pd.DataFrame):
        return rows.copy()
    return pd.DataFrame(rows)


def dataframe_to_original_row_shape(df: pd.DataFrame, original_rows):
    """
    Return DataFrame if the pipeline originally used DataFrames.
    Return list[dict] if the pipeline originally used row dictionaries.
    """
    if isinstance(original_rows, pd.DataFrame):
        return df
    return df.to_dict(orient="records")


# ---- Apply filter -----------------------------------------------------------

if DESTRUCTIVE_FILTER_ENABLED:
    clusters_df = rows_to_dataframe(loaded_tables.clusters_rows)
    mentions_df = rows_to_dataframe(loaded_tables.mentions_rows)

    print("[destructive-filter] Building foreign-anchor bank...")

    generic_anchor_keys = load_generic_anchor_keys(GENERIC_ANCHOR_BLACKLIST_PATH)

    anchors = build_anchor_bank(
        mentions=mentions_df,
        clusters=clusters_df,
        anchor_percentile=ANCHOR_PERCENTILE,
        min_anchor_mentions=MIN_ANCHOR_MENTIONS,
        generic_anchor_keys=generic_anchor_keys,
    )

    print(f"[destructive-filter] Selected anchors: {len(anchors):,}")

    filtered_mentions_df, dropped_mentions_df = filter_mentions_by_foreign_anchors(
        mentions=mentions_df,
        clusters=clusters_df,
        anchors=anchors,
        fuzzy_threshold=FUZZY_THRESHOLD,
        own_family_fuzzy_threshold=OWN_FAMILY_FUZZY_THRESHOLD,
        min_fuzzy_chars=MIN_FUZZY_CHARS,
    )

    selected_anchors_df = pd.DataFrame([
        {
            "anchor_key": anchor.key,
            "display_name": anchor.display_name,
            "n_mentions": anchor.n_mentions,
            "source_cluster_uids": "|".join(anchor.source_cluster_uids),
        }
        for anchor in anchors
    ])

    filtered_mentions_df.to_csv(FILTERED_MENTIONS_PATH, index=False)
    dropped_mentions_df.to_csv(DROPPED_MENTIONS_PATH, index=False)
    selected_anchors_df.to_csv(SELECTED_ANCHORS_PATH, index=False)

    before = len(mentions_df)
    after = len(filtered_mentions_df)
    dropped = before - after

    print("[destructive-filter] Done.")
    print(f"  Mentions before : {before:,}")
    print(f"  Mentions after  : {after:,}")
    print(f"  Dropped         : {dropped:,} ({(dropped / before * 100) if before else 0:.2f}%)")
    print(f"  Audit dropped   : {DROPPED_MENTIONS_PATH}")
    print(f"  Audit anchors   : {SELECTED_ANCHORS_PATH}")

    # This is the important object passed downstream.
    filtered_mentions_rows = dataframe_to_original_row_shape(
        filtered_mentions_df,
        loaded_tables.mentions_rows,
    )

else:
    print("[destructive-filter] Disabled.")
    filtered_mentions_rows = loaded_tables.mentions_rows

##### Global coreference cluster merging

In [ ]:
global_coref_result = merge_local_coreference_clusters(
    clusters_rows=loaded_tables.clusters_rows,
    mentions_rows=filtered_mentions_rows,
    output_dir=GLOBAL_COREF_OUTPUT_DIR,
    verbose=True,
)

print("Final global clusters:", len(global_coref_result.components))

In [ ]:
pd.read_csv(GLOBAL_COREF_OUTPUT_DIR / "global_clusters.csv")

In [ ]:
pd.read_csv(GLOBAL_COREF_OUTPUT_DIR / "global_mentions.csv")

##### Doc annotation

In [ ]:
doc = load_doc(TOKENIZED_DOC_PATH)

doc = annotate_doc_with_global_coref(
    doc=doc,
    clusters_csv=GLOBAL_COREF_OUTPUT_DIR / "global_clusters.csv",
    mentions_csv=GLOBAL_COREF_OUTPUT_DIR / "global_mentions.csv",
    verbose=True,
)

save_doc(doc, ANNOTATED_DOC_PATH)

print("[coref-annotation] Saved annotated Doc to:", ANNOTATED_DOC_PATH)
print("[coref-annotation] Summary:")
print(doc._.coref_layer.summary())

In [ ]:
coref_layer = require_coref_layer(doc)

coref_layer.summary()

In [47]:
clusters_non_singleton = [
    (cluster_id, cluster)
    for cluster_id, cluster in coref_layer.clusters.items()
    if len(cluster.mention_ids) > 1
]

clusters_sorted = sorted(
    clusters_non_singleton,
    key=lambda item: len(item[1].mention_ids),
    reverse=True,
)

print(f"Non-singleton clusters: {len(clusters_sorted)}")
print()

for cluster_id, cluster in clusters_sorted:
    print(
        f"cluster_id={cluster_id} | "
        f"canonical_name={cluster.canonical_name!r} | "
        f"n_mentions={len(cluster.mention_ids)}"
    )
    print()

Non-singleton clusters: 199

cluster_id=8 | canonical_name='Dorothy' | n_mentions=1900

cluster_id=97 | canonical_name='the Scarecrow' | n_mentions=1032

cluster_id=153 | canonical_name='the Lion' | n_mentions=837

cluster_id=48 | canonical_name='Oz' | n_mentions=801

cluster_id=111 | canonical_name='the travelers' | n_mentions=598

cluster_id=76 | canonical_name='the Wicked Witch' | n_mentions=357

cluster_id=226 | canonical_name='the Tin Woodman' | n_mentions=258

cluster_id=20 | canonical_name='Toto' | n_mentions=179

cluster_id=60 | canonical_name='the Emerald City' | n_mentions=171

cluster_id=41 | canonical_name='her friends' | n_mentions=140

cluster_id=4 | canonical_name='Aunt Em' | n_mentions=117

cluster_id=252 | canonical_name='the people' | n_mentions=99

cluster_id=67 | canonical_name='the Winkies' | n_mentions=94

cluster_id=1 | canonical_name='Kansas' | n_mentions=93

cluster_id=35 | canonical_name='the Munchkins' | n_mentions=91

cluster_id=540 | canonical_name='Glinda'